## 数据流

### 1. 前端填写数据，点击生成规划，调用 start-stream
- 生成新的 thread_id，用 uuid 生成（几乎不会重复的 ID）
- 调用 `create_conversation`，往 conversations 表插一条记录：thread_id / title / destination / create_at，status 默认 `active`，trip_plan 此时是空的

### 2. 开始跑图
- 用 `graph.astream_events` 驱动 LangGraph 执行
- checkpointer 会在**每个节点执行完之后**自动把整个 ConversationState 序列化存进 SQLite（checkpoints 表），这一步是 LangGraph 自动做的，不用手写

### 3. 图执行过程中，SSE事实推送给前端
- `on_tool_start` / `on_tool_end` 事件->推送`type:progress`(比如：正在搜索景点...)
- 节点执行顺序：`clarify -> plan -> feedback`
- **clarify**: peferences 为空 or 日期不对->interrupt,推`type:need_input`等用户
- **plan**: 跑 workflow(Phase1 ReAct Agent 调工具收集数据-> Phase2 结构化LLM生成 TripPlan)
- **feedback**: interrupt,把trip_plan 一起推送给前端(`type:need_input`,含question + trip_plan)

### 4. 图暂停(interupt)时前端拿到什么
- `stream_graph` 在 astream_events 结束后调 `graph.aget_state(config)`检查状态
- `state.tasks` 非空且有 `.interrupts`->说明图被`interrupt()`挂起
- 推送`type:need_input`,带上thread_id + question + trip_plan
- 前端把 thread_id 存起来，右侧事实预览渲染 trip_plan,输入框等用户反馈

### 5.用户继续反馈，调用 resume-stream
- 前端带上 thread_id + 用户输入文字，调`/api/chat/resume-stream`
- 后端用`Command(resume=用户输入)`恢复图，从上次interrupt的节点继续跑
- `feedback_node` 把反馈存到`last_feedback`,`should_revise`条件边判断：
  - 说了“满意/done/ok“->路由到 END
  - 有修改意见 -> 路由到revise,revice完再回feedback

## 时序图

```mermaid
sequenceDiagram
    participant FE as 前端
    participant BE as 后端 (FastAPI)
    participant AG as Agent (LangGraph)

    FE->>BE: POST /api/chat/start-stream (TripRequest)
    BE->>BE: 生成 thread_id，写入 conversations 表(status=active)
    BE->>AG: graph.astream_events(request, config)

    AG-->>BE: on_tool_start/end (搜景点/天气/酒店)
    BE-->>FE: SSE type:progress ("正在搜索景点...")

    Note over AG: clarify_node → plan_node<br/>Phase1 ReAct Agent 调工具<br/>Phase2 结构化LLM生成 TripPlan

    AG-->>BE: feedback_node 调用 interrupt(question + trip_plan)
    BE->>AG: graph.aget_state() 检查 state.tasks
    BE-->>FE: SSE type:need_input (thread_id + question + trip_plan)

    FE->>FE: 存 thread_id，渲染右侧行程预览，等待用户输入

    FE->>BE: POST /api/chat/resume-stream (thread_id + 用户反馈)
    BE->>AG: Command(resume=用户输入)

    alt 用户说"满意"
        AG->>AG: should_revise → END
        BE->>BE: on_done: complete_conversation()<br/>conversations 表 status=done, 写入 trip_plan
        BE-->>FE: SSE type:done (trip_plan)
        FE->>FE: 切换到结果页
    else 用户有修改意见
        AG->>AG: should_revise → revise_node → feedback_node → interrupt
        BE-->>FE: SSE type:need_input (更新后的 trip_plan)
        FE->>FE: 继续等待用户输入（循环）
    end

    Note over FE,AG: ── 之后点击历史记录 ──

    FE->>BE: GET /api/conversations/{thread_id}
    alt status=done
        BE->>BE: 直接从 conversations 表读 trip_plan
        BE-->>FE: trip_plan JSON
    else status=active
        BE->>AG: graph.aget_state(config) 读 checkpointer
        AG-->>BE: ConversationState.trip_plan
        BE-->>FE: trip_plan JSON
    end
```